In [ ]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import matplotlib.pyplot as plt
import cv2
import torchinfo
from pyexpat import features
from torch import nn
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import fiftyone as fo
import fiftyone.zoo as foz
import PIL.Image as Image
from jinja2.nodes import Tuple
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

#### Dataset downloading, preparation and visualization

Dataset should be around 100 images of natural photography that we need to gray scale and resize to 320x240 pixels.
The images are from CoCo train dataset and are in the folder dataset.

Homography explains how a flat image is transformed in other flat image if you look the image from different angle.

In [ ]:
class HomographyPreloadDataset(Dataset):
    def __init__(self, root = './dataset', resize=(320,240), patch_size=64, max_shift=16):
        self.images_paths = list(Path(root).glob('*.*'))
        self.resize = resize
        self.patch_size = patch_size
        self.max_shift = max_shift
        self.images = []

        print(f'Preloading {len(self.images_paths)} samples into RAM...')

        for i, image_path in enumerate(self.images_paths):
            image = self.load_image(image_path, resize)
            self.images.append(image)
            if (i + 1) % 10 == 0:
                print(f"  {i + 1}/{len(self.images_paths)} loaded...")

        print(f'Done ')


    def load_image(self, image_path, resize)-> np.ndarray:
        image = Image.open(image_path).convert('L').resize(resize)
        return np.array(image, dtype=np.float32) / 255

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        data = self.generate_image(image)
        return (
            data['sample'],
            data['offsets'],
            data['base_corners'],
            data['moved_corners'],
            data['H_mat'],
            data['warped_image'],
        )

    def get_random_top_left_corner(self, image_w, image_h) -> Tuple[int, int]:
        safe_shift = self.max_shift + 20
        x = np.random.randint(safe_shift, image_w - self.patch_size - safe_shift)
        y = np.random.randint(safe_shift, image_h - self.patch_size - safe_shift)
        return x, y

    def get_window_corners(self, x, y) -> np.array:
        top_left = [x, y]
        top_right = [x + self.patch_size -1, y]
        bottom_right = [x + self.patch_size - 1, y + self.patch_size -1]
        bottom_left = [x, y + self.patch_size - 1]
        return np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)

    def get_random_offsets(self): # todo
        return np.random.randint(-self.max_shift, self.max_shift +1, size=(4,2)).astype(np.float32)

    def generate_image(self, image):
        image_H, image_W = image.shape

        for attempt in range(5):
            x, y = self.get_random_top_left_corner(image_W, image_H)
            base = self.get_window_corners(x, y)
            offsets = self.get_random_offsets()
            moved = base + offsets

            H_mat = cv2.getPerspectiveTransform(base, moved)

            if np.linalg.cond(H_mat) > 1e10: # homography is numerically unstable
                continue

            try:
                H_mat_inv = np.linalg.inv(H_mat) # invert H to get H_inv
            except np.linalg.LinAlgError:
                continue

            warped_image = cv2.warpPerspective(image, H_mat_inv, (image_W, image_H))
            patch = image[y:y + self.patch_size, x:x + self.patch_size]
            warped_patch = warped_image[y:y + self.patch_size, x:x + self.patch_size]
            two_channel = np.stack([patch, warped_patch], axis=0)

            return {
                "sample":        torch.FloatTensor(two_channel),
                "offsets":       torch.FloatTensor(offsets),
                "base_corners":  torch.FloatTensor(base),
                "moved_corners": torch.FloatTensor(moved),
                "H_mat":         torch.FloatTensor(H_mat),
                "warped_image":  warped_image,
            }

        raise RuntimeError("Could not generate valid sample after 10 attempts")


In [ ]:
def visualize_samples(dataset, n=4):
    fig, axes = plt.subplots(n, 4, figsize=(12, n * 3))

    for i in range(n):
        image = dataset.images[i]
        data = dataset.generate_image(image)

        sample        = data['sample']
        base_corners  = data['base_corners'].numpy()
        moved_corners = data['moved_corners'].numpy()
        x = int(base_corners[0, 0])
        y = int(base_corners[0, 1])

        patch        = sample[0].numpy()
        warped_patch = sample[1].numpy()
        warped_image = data['warped_image']

        # original image with cyan square + red skewed polygon
        axes[i, 0].imshow(image, cmap='gray')
        cyan_rect = plt.Rectangle((x, y), dataset.patch_size, dataset.patch_size,linewidth=2, edgecolor='cyan', facecolor='none')
        axes[i, 0].add_patch(cyan_rect)
        red_poly = plt.Polygon(moved_corners, linewidth=2, edgecolor='red', facecolor='none', closed=True)
        axes[i, 0].add_patch(red_poly)
        axes[i, 0].set_title('Original image')
        axes[i, 0].axis('off')

        # warped image with red perfect square
        axes[i, 1].imshow(warped_image, cmap='gray')
        rect = plt.Rectangle((x, y), dataset.patch_size, dataset.patch_size,
                               linewidth=2, edgecolor='red', facecolor='none')
        axes[i, 1].add_patch(rect)
        axes[i, 1].set_title('Warped image')
        axes[i, 1].axis('off')

        # original patch (cyan region)
        axes[i, 2].imshow(patch, cmap='gray')
        axes[i, 2].set_title('Original patch')
        axes[i, 2].axis('off')

        # warped patch (red region)
        axes[i, 3].imshow(warped_patch, cmap='gray')
        axes[i, 3].set_title('Warped patch')
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
dataset = HomographyPreloadDataset()
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
visualize_samples(dataset, n=4)

#### Neural Networks

Building 2 neural networks that will share a backbone that will be built from ResNet blocks.

##### ResNet block
ResNet block is built from Conv - BatchNorm - ReLu - Conv - BatchNorm - ReLu. Its key idea is the skip connection (residual connection), where the input is added to the output of the block.
If the number of input and output channels is different, a 1×1 convolution is used to match dimensions before the addition.
This skip connection helps prevent vanishing gradients and makes training deep networks more stable. The skip connection preserves the original signal and improves gradient flow, making it easier for the network to learn useful transformations.

In [ ]:
class ResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.convolution1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.batchNorm1 = nn.BatchNorm2d(out_channels)
        self.convolution2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.batchNorm2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        if in_channels != out_channels:
            self.identity = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        else:
            self.identity = nn.Identity()

    def forward(self, x):
        identity = self.identity(x)
        out = self.convolution1(x)
        out = self.batchNorm1(out)
        out = self.relu(out)
        out = self.convolution2(out)
        out = self.batchNorm2(out)
        out += identity
        out = self.relu(out)

        return out


##### Backbone structure
The sharing backbone has 3x (ResNet block -> MaxPool) and at the end 2x ResNet blocks ant then flatten is called to convert a 3D tensor into a 1D vector so it can be fed into Linea. Linear layer compresses  the spatial features into a compact 512 dimensional feature vector that heads can use

After the convolutional layer the network has learned spatial features (specific parts in the image). The linear layer combines them all in a signal global description.
The regression or classification head will take that 512 dim summary and will predict the 8 corner offsets from it

In [ ]:
class BackboneBody(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.body = nn.Sequential(
            ResNetBlock(in_channels, 64),
            ResNetBlock(64, 64),
            nn.MaxPool2d(2),

            ResNetBlock(64, 64),
            ResNetBlock(64, 64),
            nn.MaxPool2d(2),

            ResNetBlock(64, 128),
            ResNetBlock(128, 128),
            nn.MaxPool2d(2),

            ResNetBlock(128, 128),
            ResNetBlock(128, 128),

            nn.Flatten(), # (128, 8, 8) -> (8192)
            nn.Linear(8192, 512),
            nn.ReLU(inplace=True),
        )

    def forward(self,x):
        return self.body(x)

##### Regression head
The regression head takes the 512 dimensional feature vector from the backbone and directly predicts the 8 corners offsets (4 corners x x,y) as continuoues values. While training uses MSE/RMSE loss comparing predicted offsets to the ground truth offsets that are generated during dataset creation

In [ ]:
class RegressionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = BackboneBody(2)
        self.fc = nn.Linear(512, 8) # why todo

    def forward(self, x):
        features = self.body(x)
        offsets = self.fc(features)
        return offsets

In [ ]:
regression_model = RegressionHead().to(device)
regression_optimizer = torch.optim.Adam(regression_model.parameters(), lr=0.0005)
regression_loss = nn.MSELoss()

print(torchinfo.summary(regression_model, [1, 2, 64, 64], device=device,
                        depth=4,
                        col_names=("input_size", "output_size", "num_params", "mult_adds"),
                        ))


##### Classification head
The classification head takes the 512-dimensional feature vector and instead of predicting continuous values, it treats each of the 8 offsets as a classification problem over 21 discrete bins covering the range [-16, +16]. The output is reshaped to (8, 21) and softmax is applied over the 21 bins for each offset. At inference time the final prediction is computed as a weighted sum (expected value) of the bin centers, converting the discrete distribution back into a continuous offset value. It is trained with cross-entropy loss.

In [ ]:
class ClassificationHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = BackboneBody(2)
        self.fc = nn.Linear(512, 21 * 8) # todo

    def forward(self, x):
        features = self.body(x)
        x = self.fc(features)
        x = x.view(-1, 8, 21)  #  (batch, 8 offsets, 21 bins) whatever the batch size is
        return x

In [ ]:
classification_model = ClassificationHead().to(device)
classification_optimizer = torch.optim.Adam(classification_model.parameters(), lr=0.0005)
classification_loss = nn.CrossEntropyLoss()

print(torchinfo.summary(classification_model, [1, 2, 64, 64], device=device,
                        depth=4,
                        col_names=("input_size", "output_size", "num_params", "mult_adds"),
                        ))

#### Train and result visualization
tensorboard --logdir=runs
in the terminal

In [ ]:
def train(model, loader, optimizer, loss_fn, device, step, max_steps, is_classification=False):
    model.train()
    total_loss = 0
    steps_done = 0

    for sample, offsets, base_corners, moved_corners, H_mat in loader:
        if step >= max_steps:
            break

        sample  = sample.to(device)
        offsets = offsets.to(device)

        optimizer.zero_grad()
        prediction = model(sample)

        if is_classification:
            target = offsets.view(-1, 8)
            target = torch.clamp((target + 16).long(), 0, 20)
            loss = loss_fn(prediction, target)
        else:
            target = offsets.view(-1, 8)
            prediction = prediction.view(-1, 8)
            loss = loss_fn(prediction, target)

        loss.backward()
        writer.add_scalar('Loss/homography', loss.item(), step)
        optimizer.step()

        total_loss += loss.item()
        steps_done += 1
        step += 1

    return total_loss / max(steps_done, 1), steps_done

In [ ]:
def plot_loss(loss_list, window):
    loss = np.array(loss_list, dtype=float)

    if len(loss) >= window:
        shape = (loss.size - window + 1, window)
        strides = (loss.strides[0], loss.strides[0])
        windows = np.lib.stride_tricks.as_strided(loss, shape=shape, strides=strides)
        smooth = windows.mean(axis=1)
    else:
        smooth = loss

    plt.figure(figsize=(8,4))
    plt.plot(loss, label="Raw loss")
    plt.plot(range(window-1, window-1 + len(smooth)), smooth, label=f"Smoothed (window={window})")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

##### Regression

In [ ]:
regression_losses = []
step = 0
max_steps = 50000
writer = SummaryWriter('runs/homography')

for epoch in range(1, 10000):
    train_loss, steps_done = train(
        regression_model, dataloader, regression_optimizer,
        regression_loss, device, step, max_steps,
        is_classification=False
    )
    regression_losses.append(train_loss)
    step += steps_done

    print(f'[Epoch {epoch:>3}] Step {step:>6}/{max_steps} | Loss: {train_loss:.4f}')

    if step % 10000 < len(dataloader):
        torch.save({
            'step': step,
            'model_state_dict': regression_model.state_dict(),
            'optimizer_state_dict': regression_optimizer.state_dict(),
            'losses': regression_losses,
        }, f'regression_step{step}.pth')
        print(f'           checkpoint saved at step {step}')

    if step >= max_steps:
        torch.save({
            'step': step,
            'model_state_dict': regression_model.state_dict(),
            'optimizer_state_dict': regression_optimizer.state_dict(),
            'losses': regression_losses,
        }, 'regression_final.pth')
        print('           final regression model saved')
        break

In [ ]:
def visualize_regression_results( n=4):
    regression_model.eval()

    fig, axes = plt.subplots(n, 4, figsize=(14, n * 4))

    for i in range(n):
        image = dataset.images[i]
        data = dataset.generate_image(image)

        sample        = data['sample'].unsqueeze(0).to(device)  # (1, 2, 64, 64)
        base_corners  = data['base_corners'].numpy()             # (4, 2)
        moved_corners = data['moved_corners'].numpy()            # (4, 2) true shifted
        offsets_gt    = data['offsets'].numpy()                  # (4, 2) ground truth
        patch        = data['sample'][0].numpy()  # original patch
        warped_patch = data['sample'][1].numpy()  # warped patch

        with torch.no_grad():
            pred = regression_model(sample).cpu().numpy().reshape(4, 2)  # (4, 2)

        predicted_corners = base_corners + pred  # predicted shifted corners

        # --- compute aligned patch using predicted homography ---
        H_pred = cv2.getPerspectiveTransform(
            base_corners.astype(np.float32),
            predicted_corners.astype(np.float32)
        )
        try:
            H_pred_inv = np.linalg.inv(H_pred)
            image_H, image_W = image.shape
            aligned_image = cv2.warpPerspective(image, H_pred_inv, (image_W, image_H))
            x = int(base_corners[0, 0])
            y = int(base_corners[0, 1])
            aligned_patch = aligned_image[y:y + dataset.patch_size, x:x + dataset.patch_size]
        except:
            aligned_patch = np.zeros_like(patch)

        x = int(base_corners[0, 0])
        y = int(base_corners[0, 1])

        axes[i, 0].imshow(image, cmap='gray')

        white_rect = plt.Rectangle((x, y), dataset.patch_size, dataset.patch_size,
                                    linewidth=2, edgecolor='white', facecolor='none')
        axes[i, 0].add_patch(white_rect)

        blue_poly = plt.Polygon(moved_corners, linewidth=2, edgecolor='blue', facecolor='none', closed=True)
        axes[i, 0].add_patch(blue_poly)

        red_poly = plt.Polygon(predicted_corners, linewidth=2, edgecolor='red', facecolor='none', closed=True)
        axes[i, 0].add_patch(red_poly)

        axes[i, 0].set_title('Image with predictions')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(patch, cmap='gray')
        axes[i, 1].set_title('Original')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(warped_patch, cmap='gray')
        axes[i, 2].set_title('Warped')
        axes[i, 2].axis('off')

        axes[i, 3].imshow(aligned_patch, cmap='gray')
        axes[i, 3].set_title('Aligned')
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

# The closer the red and blue quadrilaterals are, the better the prediction.
visualize_regression_results(n=4)

In [ ]:
plot_loss(regression_losses, window=50)

##### Classification

In [ ]:
classification_losses = []
step = 0

for epoch in range(1, 10000):
    train_loss, steps_done = train(
        classification_model, dataloader, classification_optimizer,
        classification_loss, device, step, max_steps,
        is_classification=True
    )
    classification_losses.append(train_loss)
    step += steps_done

    print(f'[Epoch {epoch:>3}] Step {step:>6}/{max_steps} | Loss: {train_loss:.4f}')

    if step % 10000 < len(dataloader):
        torch.save({
            'step': step,
            'model_state_dict': classification_model.state_dict(),
            'optimizer_state_dict': classification_optimizer.state_dict(),
            'losses': classification_losses,
        }, f'classification_step{step}.pth')
        print(f'           checkpoint saved at step {step}')

    if step >= max_steps:
        torch.save({
            'step': step,
            'model_state_dict': classification_model.state_dict(),
            'optimizer_state_dict': classification_optimizer.state_dict(),
            'losses': classification_losses,
        }, 'classification_final.pth')
        print('           final classification model saved')
        break

writer.close()

In [ ]:
plot_loss(classification_losses, window=50)

In [ ]:
def visualize_classification_results(n_samples=100):
    classification_model.eval()

    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    axes = axes.flatten()

    all_probabilities = [[] for _ in range(8)]
    bin_centers = np.linspace(-16, 16, 21)

    with torch.no_grad():
        for i in range(n_samples):
            image = dataset.images[i % len(dataset.images)]
            data = dataset.generate_image(image)
            sample = data['sample'].unsqueeze(0).to(device)

            prediction = classification_model(sample)                              # (1, 8, 21)
            probs = torch.softmax(prediction, dim=2)               # (1, 8, 21)
            probs = probs.squeeze(0).cpu().numpy()                  # (8, 21)

            for j in range(8):
                all_probabilities[j].append(probs[j])

    for j in range(8):
        avg_prob = np.mean(all_probabilities[j], axis=0)  # (21,)

        axes[j].bar(bin_centers, avg_prob, width=1.5, color='steelblue', alpha=0.8)
        axes[j].axvline(0, color='red', linewidth=1, linestyle='--')
        axes[j].set_title(f'Offset {j} (corner {j//2}, {"x" if j%2==0 else "y"})')
        axes[j].set_xlabel('Offset value')
        axes[j].set_ylabel('Probability')

    plt.suptitle('Classification probability distributions')
    plt.tight_layout()
    plt.show()


visualize_classification_results(n_samples=100)

#### Evaluation on different images
In training the DataLoader randomly picks images, so the same image can be picked multiple times or skipped entirely in one epoch.

In evaluation you loop directly and generate exactly 10 samples per image, so every image is used exactly 10 times, giving you a controlled and reproducible test set.
Essentially:

Training — random, repetitive, driven by DataLoader
Evaluation — systematic, every image used exactly N times

In [ ]:
test_dataset = HomographyPreloadDataset(root='./dataset_test')

def evaluate(model, dataset, device, n_per_image=10, is_classification=False):
    model.eval()
    errors = []
    bin_centers = torch.linspace(-16, 16, 21).to(device)

    with torch.no_grad():
        for image in dataset.images:                          # loop over all test images
            for _ in range(n_per_image):                      # 10 samples per image
                data = dataset.generate_image(image)

                sample     = data['sample'].unsqueeze(0).to(device)
                offsets_gt = data['offsets'].numpy().flatten()  # (8,) ground truth

                prediction = model(sample)

                if is_classification:
                    probs = torch.softmax(prediction, dim=2)
                    pred_offsets = (probs * bin_centers).sum(dim=2)
                    pred_offsets = pred_offsets.cpu().numpy().flatten()
                else:
                    pred_offsets = prediction.cpu().numpy().flatten()

                rmse = np.sqrt(np.mean((pred_offsets - offsets_gt) ** 2))
                errors.append(rmse)

    errors = np.array(errors)
    print(f"Samples: {len(errors)}")
    print(f"Mean:    {errors.mean():.4f}")
    print(f"Median:  {np.median(errors):.4f}")
    print(f"Std:     {errors.std():.4f}")
    return errors

def plot_evaluation(regression_errors, classification_errors):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # boxplot
    axes[0].boxplot([regression_errors, classification_errors], labels=['Regression', 'Classification'])
    axes[0].set_ylabel('RMSE')
    axes[0].set_title('Error distribution (boxplot)')
    axes[0].grid(True)

    # histogram
    axes[1].hist(regression_errors, bins=50, alpha=0.6, label='Regression', color='steelblue')
    axes[1].hist(classification_errors, bins=50, alpha=0.6, label='Classification', color='orange')
    axes[1].set_xlabel('RMSE')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Error distribution (histogram)')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
regression_errors = evaluate(regression_model,     test_dataset, device, is_classification=False)
classification_errors = evaluate(classification_model, test_dataset, device, is_classification=True)

plot_evaluation(regression_errors, classification_errors)

#### Comparing to classical methods

In [ ]:
def evaluate_classical(dataset, n_per_image=10):
    errors = []
    orb = cv2.ORB_create(nfeatures=1000)
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

    for image in dataset.images:
        for _ in range(n_per_image):
            data = dataset.generate_image(image)

            offsets_gt   = data['offsets'].numpy().flatten()  # (8,) ground truth
            base_corners = data['base_corners'].numpy()       # (4, 2)

            # scale patches to 256x256
            patch        = data['sample'][0].numpy()
            warped_patch = data['sample'][1].numpy()
            patch_256        = cv2.resize(patch,        (256, 256))
            warped_patch_256 = cv2.resize(warped_patch, (256, 256))

            # convert to uint8 for ORB
            patch_uint8        = (patch_256        * 255).astype(np.uint8)
            warped_uint8       = (warped_patch_256 * 255).astype(np.uint8)

            # detect keypoints and match
            kp1, des1 = orb.detectAndCompute(patch_uint8,  None)
            kp2, des2 = orb.detectAndCompute(warped_uint8, None)

            H_classical = None
            if des1 is not None and des2 is not None and len(des1) >= 4 and len(des2) >= 4:
                matches = matcher.match(des1, des2)
                if len(matches) >= 4:
                    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
                    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])
                    H_classical, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC)

            if H_classical is None:
                # assume identity if homography could not be estimated
                H_classical = np.eye(3)

            # scale corners to 256x256 space
            scale = 256 / dataset.patch_size           # 256/64 = 4
            corners_scaled = np.array([[0, 0], [255, 0], [255, 255], [0, 255]], dtype=np.float32)

            # apply homography to corners
            corners_h = cv2.perspectiveTransform(corners_scaled.reshape(1, -1, 2), H_classical)
            corners_h = corners_h.reshape(4, 2)

            # predicted offsets in 256 space, divide by 4 to get back to 64 space
            pred_offsets = ((corners_h - corners_scaled) / scale).flatten()  # (8,)

            rmse = np.sqrt(np.mean((pred_offsets - offsets_gt) ** 2))
            errors.append(rmse)

    errors = np.array(errors)
    print(f"Classical ORB — Samples: {len(errors)}")
    print(f"Mean:   {errors.mean():.4f}")
    print(f"Median: {np.median(errors):.4f}")
    print(f"Std:    {errors.std():.4f}")
    return errors

def plot_all_methods(reg_errors, cls_errors, classical_errors):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # boxplot
    axes[0].boxplot(
        [reg_errors, cls_errors, classical_errors],
        labels=['Regression', 'Classification', 'Classical ORB']
    )
    axes[0].set_ylabel('RMSE')
    axes[0].set_title('Error distribution (boxplot)')
    axes[0].grid(True)

    # histogram
    axes[1].hist(reg_errors,       bins=50, alpha=0.6, label='Regression',       color='steelblue')
    axes[1].hist(cls_errors,       bins=50, alpha=0.6, label='Classification',    color='orange')
    axes[1].hist(classical_errors, bins=50, alpha=0.6, label='Classical ORB',     color='green')
    axes[1].set_xlabel('RMSE')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Error distribution (histogram)')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
classical_errors = evaluate_classical(test_dataset, n_per_image=10)
plot_all_methods(regression_errors, classification_errors, classical_errors)
